# Análisis Inteligente del Consumo Energético

El objetivo de este proyecto es desarrollar un modelo de Machine Learning capaz de clasificar el nivel de eficiencia energética de una vivienda o pequeño establecimiento a partir de diferentes características relacionadas con el consumo eléctrico.

Se generará un dataset sintético que represente distintos perfiles de consumo.

Posteriormente se realizará un análisis exploratorio de los datos (EDA), el entrenamiento de modelos supervisados y la evaluación de su desempeño para seleccionar el modelo más adecuado.

# Variables del Dataset

Para representar el comportamiento energético de una vivienda se seleccionaron las siguientes variables.

1. Fecha

      Tipo: Fecha (YYYY-MM)

      -Representa el período al que corresponde el consumo energético registrado.

      Permite realizar análisis temporales como:

      -Evolución del consumo.

      -Comparación entre meses.

      -Tendencias de eficiencia.

      -Variaciones estacionales.

2. Consumo mensual (kWh)

      Tipo: Numérica

      Representa la cantidad de energía consumida durante un mes.

      Es la variable principal del problema y será utilizada para:

      -Estimar el costo mensual.

      -Analizar patrones de consumo.

      -Detectar consumos elevados.
      
3. Uso en horario pico

      Tipo: Booleana

      Indica si la mayor parte del consumo ocurre durante los horarios de mayor demanda eléctrica.

      Su análisis permite:

      -Detectar hábitos poco eficientes.

      -Generar recomendaciones para redistribuir el consumo.

4. Horas de alto consumo

      Tipo: Numérica

      Cantidad aproximada de horas diarias durante las cuales el consumo eléctrico es elevado.

      Permite identificar:

      -Intensidad del uso energético.

      -Relación entre tiempo de uso y consumo mensual.

5. Cantidad de equipos eléctricos

      Tipo: Numérica

      Número de electrodomésticos o equipos eléctricos utilizados regularmente.

      -Esta variable permite analizar cómo influye la cantidad de dispositivos sobre el consumo energético.

6. Cantidad de personas

      Tipo: Numérica

      -Cantidad de personas que habitan el inmueble.

      Se incorpora debido a que el consumo suele aumentar con el número de habitantes.

7. Tipo de inmueble

      Tipo: Categórica

      Puede tomar los valores:

      -Casa

      -Departamento

      -Comercio

      Permite analizar diferencias de consumo entre distintos tipos de establecimientos.

8. Categoría de eficiencia energética (Target)

      Tipo: Categórica

      Representa la clasificación que el modelo deberá aprender.

      Se utilizarán tres categorías:

      -Eficiente

      -Moderado
      
      -Ineficiente

      Las etiquetas serán generadas mediante un conjunto de reglas que consideran el consumo energético y otros factores relacionados con los hábitos de uso.

# Generación del Dataset

Se desarrolló un generador de datos sintéticos.

El objetivo es representar distintos perfiles de consumo energético manteniendo relaciones coherentes entre las variables.

Se generarán 1000 registros correspondientes a distintos inmuebles con características variadas.

Posteriormente estas observaciones serán utilizadas para entrenar y evaluar modelos supervisados de clasificación.

In [ ]:
import random
import pandas as pd
from datetime import datetime

NUM_REGISTROS = 1000

TIPOS_INMUEBLE = [
    "Casa",
    "Departamento",
    "Comercio"
]
def generar_fecha():
    año = random.randint(2023, 2025)
    mes = random.randint(1, 12)
    return datetime(año, mes, 1).strftime("%Y-%m")


def generar_tipo():
    return random.choices(
        TIPOS_INMUEBLE,
        weights=[50, 35, 15]
    )[0]


def generar_personas(tipo):

    if tipo == "Casa":
        return random.randint(2, 6)

    elif tipo == "Departamento":
        return random.randint(1, 4)

    return random.randint(2, 8)


def generar_equipos(personas):
    return random.randint(personas + 2, personas + 10)


def generar_horas():
    return random.randint(2, 12)


def generar_horario_pico():
    return random.random() < 0.65


def calcular_consumo(personas, equipos, horas, horario_pico):

    consumo = 80

    consumo += personas * random.randint(20, 35)

    consumo += equipos * random.randint(6, 10)

    consumo += horas * random.randint(8, 15)

    if horario_pico:
        consumo += random.randint(20, 50)

    consumo += random.randint(-15, 15)

    return max(consumo, 100)

#INdice para calcular categoria

def calcular_indice(consumo,
                    horario_pico,
                    horas,
                    equipos,
                    personas):

    indice = 100

    if consumo > 600:
        indice -= 40
    elif consumo > 450:
        indice -= 25
    elif consumo > 300:
        indice -= 10

    if horario_pico:
        indice -= 15

    if horas >= 10:
        indice -= 15
    elif horas >= 7:
        indice -= 8


    if equipos >= 15:
        indice -= 15
    elif equipos >= 10:
        indice -= 8


    if personas >= 5:
        indice -= 5

    indice += random.randint(-8, 8)

    indice = max(0, min(indice, 100))

    return indice

def clasificar(indice):

    if indice >= 75:
        return "Eficiente"

    elif indice >= 45:
        return "Moderado"

    return "Ineficiente"


def generar_registro():

    tipo = generar_tipo()

    personas = generar_personas(tipo)

    equipos = generar_equipos(personas)

    horas = generar_horas()

    horario_pico = generar_horario_pico()

    consumo = calcular_consumo(
        personas,
        equipos,
        horas,
        horario_pico
    )

    indice = calcular_indice(
        consumo,
        horario_pico,
        horas,
        equipos,
        personas
    )

    categoria = clasificar(indice)

    return {
        "fecha": generar_fecha(),
        "consumo_kwh": consumo,
        "uso_horario_pico": horario_pico,
        "horas_alto_consumo": horas,
        "cantidad_equipos": equipos,
        "cantidad_personas": personas,
        "tipo_inmueble": tipo,
        "categoria": categoria
    }
c
dataset = []

for _ in range(NUM_REGISTROS):
    dataset.append(generar_registro())

df = pd.DataFrame(dataset)

df.to_csv(
    "consumo_energetico.csv",
    index=False,
    encoding="utf-8"
)

print(df.head())

print("\nDistribución de categorías:")
print(df["categoria"].value_counts())

     fecha  consumo_kwh  uso_horario_pico  horas_alto_consumo  \
0  2024-06          358             False                   3   
1  2024-08          336              True                   2   
2  2023-05          331              True                   3   
3  2023-10          574              True                  11   
4  2023-10          309             False                   9   

   cantidad_equipos  cantidad_personas tipo_inmueble    categoria  
0                15                  5          Casa     Moderado  
1                12                  3  Departamento     Moderado  
2                 8                  3  Departamento    Eficiente  
3                13                  6          Casa  Ineficiente  
4                 7                  2          Casa    Eficiente  

Distribución de categorías:
categoria
Moderado       554
Eficiente      363
Ineficiente     83
Name: count, dtype: int64


#Conociendo el Data Set

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("consumo_energetico.csv")

In [ ]:
print("Primeros registros")
df.head()

Primeros registros


,fecha,consumo_kwh,uso_horario_pico,horas_alto_consumo,cantidad_equipos,cantidad_personas,tipo_inmueble,categoria
0,2024-06,358,False,3,15,5,Casa,Moderado
1,2024-08,336,True,2,12,3,Departamento,Moderado
2,2023-05,331,True,3,8,3,Departamento,Eficiente
3,2023-10,574,True,11,13,6,Casa,Ineficiente
4,2023-10,309,False,9,7,2,Casa,Eficiente


In [ ]:
print("Últimos registros")
df.tail()

Últimos registros


,fecha,consumo_kwh,uso_horario_pico,horas_alto_consumo,cantidad_equipos,cantidad_personas,tipo_inmueble,categoria
995,2025-08,239,True,5,3,1,Departamento,Eficiente
996,2024-02,385,False,8,9,6,Casa,Eficiente
997,2024-05,290,True,3,10,2,Casa,Eficiente
998,2025-08,444,False,11,12,6,Casa,Moderado
999,2024-12,361,True,7,8,4,Comercio,Moderado


In [ ]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas: 1000
Columnas: 8


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   fecha               1000 non-null   object
 1   consumo_kwh         1000 non-null   int64 
 2   uso_horario_pico    1000 non-null   bool  
 3   horas_alto_consumo  1000 non-null   int64 
 4   cantidad_equipos    1000 non-null   int64 
 5   cantidad_personas   1000 non-null   int64 
 6   tipo_inmueble       1000 non-null   object
 7   categoria           1000 non-null   object
dtypes: bool(1), int64(4), object(3)
memory usage: 55.8+ KB


In [ ]:
df.describe()

,consumo_kwh,horas_alto_consumo,cantidad_equipos,cantidad_personas
count,1000.000000,1000.000000,1000.00000,1000.000000
mean,355.853000,6.839000,9.55800,3.579000
std,79.820365,3.222959,3.14683,1.678266
min,154.000000,2.000000,3.00000,1.000000
25%,297.750000,4.000000,7.00000,2.000000
50%,352.500000,7.000000,10.00000,3.000000
75%,407.250000,10.000000,12.00000,5.000000
max,657.000000,12.000000,18.00000,8.000000


In [ ]:
print("Consumo promedio")

df["consumo_kwh"].mean()



Consumo promedio
Consumo mínimo
154
Consumo máximo
657


In [ ]:
print("Consumo mínimo")
print(df["consumo_kwh"].min())

print("Consumo máximo")
print(df["consumo_kwh"].max())

Consumo mínimo
154
Consumo máximo
657


In [ ]:
TARIFA = 0.75

df["costo_estimado"] = df["consumo_kwh"] * TARIFA

In [ ]:
print("Costo promedio mensual")

df["costo_estimado"].mean()

Costo promedio mensual


np.float64(266.88975)

In [ ]:
df["costo_estimado"].describe()

,costo_estimado
count,1000.000000
mean,266.889750
std,59.865274
min,115.500000
25%,223.312500
50%,264.375000
75%,305.437500
max,492.750000


In [ ]:
df["categoria"].value_counts()

,count
categoria,
Moderado,554
Eficiente,363
Ineficiente,83


In [ ]:
round(
    df["categoria"].value_counts(normalize=True)*100,
    2
)

,proportion
categoria,
Moderado,55.4
Eficiente,36.3
Ineficiente,8.3


In [ ]:
df["tipo_inmueble"].value_counts()

,count
tipo_inmueble,
Casa,512
Departamento,355
Comercio,133


In [ ]:
round(
    df["tipo_inmueble"].value_counts(normalize=True)*100,
    2
)

,proportion
tipo_inmueble,
Casa,51.2
Departamento,35.5
Comercio,13.3


In [ ]:
df["uso_horario_pico"].value_counts()

,count
uso_horario_pico,
True,670
False,330


In [ ]:
round(
    df["uso_horario_pico"].value_counts(normalize=True)*100,
    2
)

,proportion
uso_horario_pico,
True,67.0
False,33.0


In [ ]:
df.groupby("tipo_inmueble")["consumo_kwh"].mean().sort_values(ascending=False)

,consumo_kwh
tipo_inmueble,
Comercio,404.684211
Casa,368.298828
Departamento,319.608451


In [ ]:
df.groupby("categoria")["consumo_kwh"].mean()

,consumo_kwh
categoria,
Eficiente,289.639118
Ineficiente,500.433735
Moderado,377.577617
